In [1]:
%load_ext autoreload
%autoreload 2

In [31]:
from data_collection import build_dataset
from data_selection import select_data, split_data, create_yolo_format, convert_labels_to_numeric_representation
import pandas as pd
from ultralytics import YOLO

Function which pulls and organizes the images. Starts with pulling all the pokemon api, which takes the longest. Followed by kaggle then huggingface. On my machine it takes approximtely 3-4 minutes

In [8]:
build_dataset()

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Data collection completed


In [9]:
df = select_data(sources='all', levels=[1])

Selected 2219 images across 12 Pokemon.


In [10]:
train, test, val = split_data(df)

print(train)

                                             image_path       label  \
100   /home/branchn/.Gtech/Pokemon-Visualization-Too...    Magikarp   
776   /home/branchn/.Gtech/Pokemon-Visualization-Too...  Butterfree   
1458  /home/branchn/.Gtech/Pokemon-Visualization-Too...     Scyther   
1699  /home/branchn/.Gtech/Pokemon-Visualization-Too...       Doduo   
2079  /home/branchn/.Gtech/Pokemon-Visualization-Too...  Butterfree   
...                                                 ...         ...   
1949  /home/branchn/.Gtech/Pokemon-Visualization-Too...      Meowth   
2126  /home/branchn/.Gtech/Pokemon-Visualization-Too...     Pikachu   
1264  /home/branchn/.Gtech/Pokemon-Visualization-Too...  Kangaskhan   
865   /home/branchn/.Gtech/Pokemon-Visualization-Too...  Butterfree   
1812  /home/branchn/.Gtech/Pokemon-Visualization-Too...     Snorlax   

           source  
100    PokemonAPI  
776    PokemonAPI  
1458       Kaggle  
1699  HuggingFace  
2079  HuggingFace  
...           ...  
1949  H

This following code puts the data into a Ultralytics format for yolo models. It essentially tags each image with a bounding box (the whole image is the bounding box) needed for yolo models, and stores it in a text file. The images are also copied to keep referencing in an ultralytics yaml easier. 


In [64]:
list_of_labels = None #["magikarp","pikachu", "meowth"]
print((df["label"].value_counts()))

# list_of_labels = ["butterfree", "chansey", "doduo"]
numberic_labels_train = convert_labels_to_numeric_representation(train, list_of_labels)
numberic_labels_test = convert_labels_to_numeric_representation(test, list_of_labels)
numberic_labels_val = convert_labels_to_numeric_representation(val, list_of_labels)

# print(numberic_labels_train)
# print(numberic_labels_test)
# print(numberic_labels_val)

create_yolo_format(train, 'train', numberic_labels_train)
create_yolo_format(test, 'test', numberic_labels_test)
create_yolo_format(val, 'val', numberic_labels_val)

label
Pikachu       218
Scyther       213
Magikarp      202
Butterfree    197
Doduo         193
Snorlax       182
Lapras        176
Kangaskhan    173
Pidgey        170
Chansey       166
Eevee         166
Meowth        163
Name: count, dtype: int64
Label to numeric mapping: {'butterfree': 0, 'chansey': 1, 'doduo': 2, 'eevee': 3, 'kangaskhan': 4, 'lapras': 5, 'magikarp': 6, 'meowth': 7, 'pidgey': 8, 'pikachu': 9, 'scyther': 10, 'snorlax': 11}
Label to numeric mapping: {'butterfree': 0, 'chansey': 1, 'doduo': 2, 'eevee': 3, 'kangaskhan': 4, 'lapras': 5, 'magikarp': 6, 'meowth': 7, 'pidgey': 8, 'pikachu': 9, 'scyther': 10, 'snorlax': 11}
Label to numeric mapping: {'butterfree': 0, 'chansey': 1, 'doduo': 2, 'eevee': 3, 'kangaskhan': 4, 'lapras': 5, 'magikarp': 6, 'meowth': 7, 'pidgey': 8, 'pikachu': 9, 'scyther': 10, 'snorlax': 11}
Creating YOLO format text files for training data...


Creating yolo data for train : 100%|██████████| 1775/1775 [00:00<00:00, 6494.74it/s]


Creating YOLO format text files for testing data...


Creating yolo data for test : 100%|██████████| 222/222 [00:00<00:00, 6578.42it/s]


Creating YOLO format text files for validation data...


Creating yolo data for val : 100%|██████████| 222/222 [00:00<00:00, 6533.37it/s]


In [65]:
model = YOLO('yolov8n.pt')
model.train(data='../Dataset/My_yolo_dataset/pokedata.yaml', epochs=20, imgsz=640, device=[4,5], lr0=0.005)



New https://pypi.org/project/ultralytics/8.4.34 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.33 🚀 Python-3.11.5 torch-2.10.0+cu126 CUDA:4 (NVIDIA H100 PCIe, 81110MiB)
                                                      CUDA:5 (NVIDIA H100 PCIe, 81110MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../Dataset/My_yolo_dataset/pokedata.yaml, degrees=0.0, deterministic=True, device=4,5, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=3

In [66]:
results = model.predict(source="../Dataset/My_yolo_dataset/test/images/", project="predictions/", name="test_predictions", conf=0.25, save=True , exist_ok=True)



image 1/185 /home/branchn/.Gtech/Pokemon-Visualization-Tool/data_operations/../Dataset/My_yolo_dataset/test/images/00000029_jpg.rf.efabd6ace535ea180bb62af8b2df9553.jpg: 640x640 1 pikachu, 5.8ms
image 2/185 /home/branchn/.Gtech/Pokemon-Visualization-Tool/data_operations/../Dataset/My_yolo_dataset/test/images/00000046_jpg.rf.b19e1ad9be78b9cb1d818f39c934e8f5.jpg: 640x640 1 pikachu, 6.0ms
image 3/185 /home/branchn/.Gtech/Pokemon-Visualization-Tool/data_operations/../Dataset/My_yolo_dataset/test/images/00000072_jpeg.rf.672e9cfa83ccc6298f95b69087ec570d.jpg: 640x640 1 pikachu, 5.8ms
image 4/185 /home/branchn/.Gtech/Pokemon-Visualization-Tool/data_operations/../Dataset/My_yolo_dataset/test/images/01-jpgb9bd6bd6-3a57-4dc9-8c6e-fda18d6f9004Large_jpg.rf.1b6794f6b4c72b8631571a6af9f18b39.jpg: 640x640 1 meowth, 5.8ms
image 5/185 /home/branchn/.Gtech/Pokemon-Visualization-Tool/data_operations/../Dataset/My_yolo_dataset/test/images/192479c116a0451e896c8cd10202fe69_jpg.rf.ab1fc58455846296dc6f236a90fa8